In [1]:
"""
report1_data_quality.py  —  Modules 1-5 Data Quality Checks
=============================================================
Output: Reports/data_quality_report.xlsx

Sheets:
  00_Summary    traffic-light overview of every check
  01_Schema     column presence and data type validation
  02_Missing    missing value counts per column
  03_Duplicates duplicate row detection with examples
  04_Values     negative / zero / future date checks
  05_Units      commodities using more than one unit of measure

Usage:
  python3 report1_data_quality.py
"""

import glob
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Environment detection (Colab vs Local) ────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    import os as _os
    _os.listdir('/content/drive/MyDrive')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data/Dataset')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data/Exports')
    BASE_DIR = Path('/content/drive/MyDrive/Dashboard-BR-CA-Data/Dataset')
    IN_COLAB = True
except Exception:
    BASE_DIR = Path.cwd().parent / 'Dataset'
    IN_COLAB = False

EXPORTS_DIR  = BASE_DIR / 'Exports'
NORM_FILE    = BASE_DIR / 'Normalised' / 'prices_normalised.csv'
REPORTS_DIR  = BASE_DIR / 'Reports'
LOOKER_DIR   = BASE_DIR / 'Looker'

print(f'Environment : {"Colab" if IN_COLAB else "Local"}')
print(f'Base dir    : {BASE_DIR}')
print(f'Exists      : {BASE_DIR.exists()}')


OUT_FILE    = REPORTS_DIR / "data_quality_report.xlsx"

EXPECTED_COLUMNS   = ["Period","Commodity","Province","Country",
                       "State","Value ($)","Quantity","Unit of measure"]
KEY_COLS_DUPLICATE = ["Period","Commodity","Province","Country","State"]
TODAY = pd.Timestamp.now().normalize()

# colours
HDR="1F3864"; WHT="FFFFFF"; ALT="EEF4FB"; NTE="DEEAF1"
OK_B="C6EFCE"; OK_F="276221"; WN_B="FFEB9C"; WN_F="9C5700"
FL_B="FFC7CE"; FL_F="9C0006"

def _fill(c): return PatternFill("solid",start_color=c,end_color=c)
def _font(bold=False,color="000000",sz=10): return Font(name="Arial",bold=bold,color=color,size=sz)
def _border():
    s=Side(style="thin",color="BDD7EE"); return Border(left=s,right=s,top=s,bottom=s)
def _left():   return Alignment(horizontal="left",  vertical="center",wrap_text=True)
def _center(): return Alignment(horizontal="center",vertical="center",wrap_text=True)

def _status_style(v):
    s=str(v).upper()
    if "OK" in s:   return OK_B,OK_F
    if "WARN" in s: return WN_B,WN_F
    return FL_B,FL_F

def _title(ws,text,ncols):
    ws.append([text]+[""]*(ncols-1)); r=ws.max_row
    ws.cell(r,1).font=_font(bold=True,color=WHT,sz=12)
    ws.cell(r,1).fill=_fill(HDR); ws.cell(r,1).alignment=_left()
    ws.row_dimensions[r].height=28
    if ncols>1: ws.merge_cells(start_row=r,start_column=1,end_row=r,end_column=ncols)

def _note(ws,text,ncols):
    ws.append([text]+[""]*(ncols-1)); r=ws.max_row
    ws.cell(r,1).font=_font(color="1F3864"); ws.cell(r,1).fill=_fill(NTE)
    ws.cell(r,1).alignment=_left()
    if ncols>1: ws.merge_cells(start_row=r,start_column=1,end_row=r,end_column=ncols)

def _hdr_row(ws,cols,widths=None):
    ws.append(cols); r=ws.max_row
    for c in range(1,len(cols)+1):
        cell=ws.cell(r,c)
        cell.font=_font(bold=True,color=WHT); cell.fill=_fill(HDR)
        cell.alignment=_center(); cell.border=_border()
    if widths:
        for c,w in enumerate(widths,1): ws.column_dimensions[get_column_letter(c)].width=w
    ws.row_dimensions[r].height=30
    return r

def _data_row(ws,vals,alt=False,status_col=None):
    ws.append(vals); r=ws.max_row; bg=ALT if alt else WHT
    for c,v in enumerate(vals,1):
        cell=ws.cell(r,c); cell.border=_border(); cell.alignment=_left()
        if c==status_col and isinstance(v,str):
            b,f=_status_style(v); cell.fill=_fill(b); cell.font=_font(bold=True,color=f)
        else:
            cell.fill=_fill(bg); cell.font=_font()

# load
def _load_one(fp):
    df=pd.read_csv(fp,skiprows=1,dtype=str); df.columns=df.columns.str.strip()
    if "Period" in df.columns:
        df=df[pd.to_datetime(df["Period"],errors="coerce").notna()].copy()
    df["_source_file"]=Path(fp).name; return df

def load_all(files):
    frames,errors=[],[]
    for f in files:
        except Exception as e: errors.append({"file":Path(f).name,"error":str(e)})
    return (pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()),errors

# checks
def check_schema(raw,load_errors):
    actual=[c for c in raw.columns if c!="_source_file"]
    missing=[c for c in EXPECTED_COLUMNS if c not in actual]
    extra=[c for c in actual if c not in EXPECTED_COLUMNS]
    rows=[
        {"Check":"Files loaded without errors","Expected":"0 errors",
         "Found":f"{len(load_errors)} errors",
         "Status":"✅ OK" if not load_errors else "❌ FAIL",
         "Detail":"; ".join(e["file"] for e in load_errors[:5]) or "—"},
        {"Check":"All expected columns present","Expected":str(len(EXPECTED_COLUMNS)),
         "Found":str(len(actual)),"Status":"✅ OK" if not missing else "❌ FAIL",
         "Detail":", ".join(missing) or "—"},
        {"Check":"No unexpected extra columns","Expected":"None",
         "Found":", ".join(extra) or "None",
         "Status":"✅ OK" if not extra else "⚠️ WARN","Detail":"—"},
    ]
    for col,dtype in [("Period","Date"),("Value ($)","Numeric"),("Quantity","Numeric")]:
        raw_s=raw.get(col,pd.Series(dtype=str)).fillna("")
        exists=raw_s.astype(str).str.strip()!=""
        parsed=(pd.to_datetime(raw_s,errors="coerce").notna() if dtype=="Date"
                else pd.to_numeric(raw_s,errors="coerce").notna())
        fails=int((exists&~parsed).sum())
        rows.append({"Check":f"Data type valid: {col}","Expected":dtype,
                     "Found":f"{fails:,} parse failures" if fails else "All valid",
                     "Status":"✅ OK" if fails==0 else "❌ FAIL","Detail":"—"})
    return pd.DataFrame(rows)

def check_missing(raw):
    total=len(raw); rows=[]
    for col in EXPECTED_COLUMNS:
        if col not in raw.columns:
            rows.append({"Column":col,"Missing":"N/A","% of Total":"N/A",
                         "Status":"❌ COLUMN MISSING","Note":""}); continue
        n=int(raw[col].isna().sum()+(raw[col].astype(str).str.strip()=="").sum())
        p=round(n/total*100,2) if total else 0
        if col=="State":
            s="⚠️ WARN" if n>0 else "✅ OK"; note="Optional — non-US destinations have no state"
        else:
            s="✅ OK" if n==0 else ("⚠️ WARN" if p<1 else "❌ FAIL"); note=""
        rows.append({"Column":col,"Missing":n,"% of Total":p,"Status":s,"Note":note})
    return pd.DataFrame(rows)

def check_duplicates(raw):
    key=[c for c in KEY_COLS_DUPLICATE if c in raw.columns]
    n_dup=int(raw.duplicated(subset=key).sum()); total=len(raw)
    summary=pd.DataFrame([{
        "Key Columns":", ".join(key),"Total Rows":f"{total:,}",
        "Duplicate Rows":f"{n_dup:,}",
        "% of Total":f"{n_dup/total*100:.2f}%" if total else "0%",
        "Status":"✅ OK" if n_dup==0 else "❌ FAIL",
        "Note":"May mean a file was loaded twice" if n_dup else "—"}])
    examples=(raw[raw.duplicated(subset=key,keep=False)]
                 .sort_values(key).head(50)[key+["_source_file"]]
              if n_dup else pd.DataFrame())
    return summary,examples

def check_values(raw):
    df=raw.copy()
    df["_v"]=pd.to_numeric(df.get("Value ($)"),errors="coerce")
    df["_q"]=pd.to_numeric(df.get("Quantity"), errors="coerce")
    df["_d"]=pd.to_datetime(df.get("Period"),  errors="coerce")
    df["_unit"]=df.get("Unit of measure",pd.Series("")).astype(str).str.strip().str.lower()
    total=len(df)
    def r(check,n,note):
        p=round(n/total*100,2) if total else 0
        s="✅ OK" if n==0 else ("⚠️ WARN" if any(k in check for k in ["Zero","Future"]) else "❌ FAIL")
        return {"Check":check,"Count":int(n),"% of Total":p,"Status":s,"Note":note}
    blank=(df["_unit"]=="blank")
    return pd.DataFrame([
        r("Negative Value ($)",        int((df["_v"]<0).sum()),                          "Should never be negative"),
        r("Negative Quantity",          int((df["_q"]<0).sum()),                          "Should never be negative"),
        r("Zero Qty — Blank unit",      int(((df["_q"]==0) & blank).sum()),               "Expected — Statistics Canada omits qty for these HS codes. Handled in Module 6a"),
        r("Zero Qty — Real unit (⚠️)",  int(((df["_q"]==0) & ~blank).sum()),              "Suspicious — a unit is specified but qty is 0. May be a data entry error"),
        r("Unparseable dates",           int(df["_d"].isna().sum()),                       ""),
        r("Future dates",                int((df["_d"]>TODAY).sum()),                      "May indicate projected or mis-dated data"),
    ])


# Units that are count-based — zero qty is highly suspicious (you can't export 0 bicycles)
_COUNT_UNITS = {"number","number of dozens","number of gross","number of pairs",
                "number of sets","number of items"}
# Weight/volume — zero qty may be a rounding artefact for very small shipments
_WEIGHT_UNITS = {"weight in kilograms","weight in metric tonne","weight in grams",
                 "weight in milligrams","weight in carats","weight in milligrams of active ingredient"}
_VOLUME_UNITS = {"volume in litres","volume in cubic metres",
                 "volume in litres of pure alcohol"}

def _unit_flag(unit: str) -> str:
    u = unit.strip().lower()
    if u in _COUNT_UNITS:   return "⚠️ Count unit — very suspicious"
    if u in _WEIGHT_UNITS:  return "Weight — possible rounding"
    if u in _VOLUME_UNITS:  return "Volume — possible rounding"
    return "Other"


def zero_qty_commodity_table(raw):
    """
    Focuses only on rows where Qty=0, unit is NOT blank, and Value > 0.
    These rows have export revenue but no quantity — unit price cannot be
    calculated (÷ zero) so they are excluded from price analysis.

    Returns:
      commodity_summary  — one row per affected commodity with unit and flag
      example_rows       — up to 20 individual rows sorted by commodity then value desc
    """
    df=raw.copy()
    if df.empty or "Commodity" not in df.columns:
        return pd.DataFrame(), pd.DataFrame()
    df["_q"]    =pd.to_numeric(df.get("Quantity"),  errors="coerce")
    df["_v"]    =pd.to_numeric(df.get("Value ($)"), errors="coerce")
    df["_unit"] =df.get("Unit of measure",pd.Series("")).astype(str).str.strip().str.lower()
    df["_blank"]=df["_unit"]=="blank"

    mask=(df["_q"]==0) & ~df["_blank"] & (df["_v"]>0)
    suspect=df[mask].copy()
    if suspect.empty:
        return pd.DataFrame(), pd.DataFrame()

    total_per_comm=df.groupby("Commodity").size().rename("Total Rows in Commodity")

    grp=(suspect.groupby("Commodity")
         .agg(Rows=("_v","count"),
              Total_Value=("_v","sum"),
              Countries=("Country","nunique"),
              Unit=("_unit", lambda x: x.value_counts().index[0]))  # dominant unit
         .reset_index()
         .merge(total_per_comm, on="Commodity", how="left"))

    grp["% of Commodity Rows"]=round(grp["Rows"]/grp["Total Rows in Commodity"]*100,1)
    grp["Total_Value"]=grp["Total_Value"].round(0).astype(int)
    grp["Unit"]=grp["Unit"].str.title()
    grp["Unit Flag"]=grp["Unit"].str.lower().apply(_unit_flag)

    grp=(grp.rename(columns={
              "Rows":"Rows (Qty=0, Value>0)",
              "Total_Value":"Total Value $ (excluded)"})
           .sort_values(["Rows (Qty=0, Value>0)","Total Value $ (excluded)"],ascending=False)
           .reset_index(drop=True)
           [["Commodity","Total Rows in Commodity","Rows (Qty=0, Value>0)",
             "% of Commodity Rows","Countries","Unit","Unit Flag",
             "Total Value $ (excluded)"]])

    ex_cols=["Commodity","Period","Country","Quantity","Unit of measure","Value ($)","_source_file"]
    ex_cols=[c for c in ex_cols if c in df.columns]
    examples=(suspect.sort_values(["Commodity","_v"],ascending=[True,False])
                     .head(20)[ex_cols])

    return grp, examples

def check_units(raw):
    if raw.empty or "Commodity" not in raw.columns:
        return pd.DataFrame([{"Total Commodities":0,"Multi-Unit Commodities":0,
                               "% Affected":0,"Status":"⚠️ WARN","Note":"No data loaded"}]),pd.DataFrame()
    df=raw.copy()
    df["_u"]=df.get("Unit of measure",pd.Series("")).astype(str).str.strip().str.lower()
    s=df.groupby("Commodity")["_u"].nunique().reset_index(name="distinct_units")
    total=len(s); inc=s[s["distinct_units"]>1]; n=len(inc)
    summary=pd.DataFrame([{
        "Total Commodities":total,"Multi-Unit Commodities":n,
        "% Affected":round(n/total*100,2) if total else 0,
        "Status":"✅ OK" if n==0 else "⚠️ WARN",
        "Note":"Some variation is expected (e.g. MBq and GBq — both handled in 6a)"}])
    detail=pd.DataFrame()
    if n:
        d=(df.groupby(["Commodity","_u"]).size().reset_index(name="Rows")
             .merge(s,on="Commodity").rename(columns={"_u":"Unit"})
             .query("distinct_units>1")
             .sort_values(["distinct_units","Rows"],ascending=False)
             .head(100)[["Commodity","Unit","Rows","distinct_units"]]
             .rename(columns={"distinct_units":"Distinct Units"}))
        detail=d
    return summary,detail

# build
def build(raw,load_errors):
    schema_df        =check_schema(raw,load_errors)
    missing_df       =check_missing(raw)
    dup_sum,dup_ex   =check_duplicates(raw)
    values_df        =check_values(raw)
    zero_top10,zero_susp=zero_qty_commodity_table(raw)
    unit_sum,unit_det=check_units(raw)
    nfiles=raw["_source_file"].nunique() if not raw.empty and "_source_file" in raw.columns else 0
    run_dt=datetime.now().strftime("%Y-%m-%d %H:%M")

    wb=Workbook(); wb.remove(wb.active)

    # 00 Summary
    ws=wb.create_sheet("00_Summary"); ws.sheet_view.showGridLines=False
    _title(ws,"DATA QUALITY REPORT — CANADIAN EXPORT DATA",5); ws.append([""])
    _note(ws,f"Generated: {run_dt}   |   Files: {nfiles:,}   |   Total rows: {len(raw):,}",5)
    ws.append([""])
    hr=_hdr_row(ws,["Module","Check","Found","Status","Detail"],[22,36,26,14,46])
    for i,(_, r) in enumerate(schema_df.iterrows()):
        _data_row(ws,["1 — Schema",r["Check"],r["Found"],r["Status"],r["Detail"]],alt=i%2==1,status_col=4)
    for i,(_, r) in enumerate(missing_df.iterrows()):
        _data_row(ws,["2 — Missing",r["Column"],f"{r['Missing']} ({r['% of Total']}%)",r["Status"],r["Note"]],alt=i%2==1,status_col=4)
    for i,(_, r) in enumerate(dup_sum.iterrows()):
        _data_row(ws,["3 — Duplicates","Duplicate rows",r["Duplicate Rows"],r["Status"],r["Note"]],alt=i%2==1,status_col=4)
    for i,(_, r) in enumerate(values_df.iterrows()):
        _data_row(ws,["4 — Values",r["Check"],f"{r['Count']:,} ({r['% of Total']}%)",r["Status"],r["Note"]],alt=i%2==1,status_col=4)
    for i,(_, r) in enumerate(unit_sum.iterrows()):
        _data_row(ws,["5 — Units","Multi-unit commodities",r["Multi-Unit Commodities"],r["Status"],r["Note"]],alt=i%2==1,status_col=4)
    ws.freeze_panes=f"A{hr+1}"

    # 01 Schema
    ws1=wb.create_sheet("01_Schema"); ws1.sheet_view.showGridLines=False
    _title(ws1,"MODULE 1 — SCHEMA VALIDATION",5); ws1.append([""])
    hr1=_hdr_row(ws1,["Check","Expected","Found","Status","Detail"],[34,20,26,14,44])
    for i,(_, r) in enumerate(schema_df.iterrows()):
        _data_row(ws1,[r["Check"],r["Expected"],r["Found"],r["Status"],r["Detail"]],alt=i%2==1,status_col=4)
    ws1.freeze_panes=f"A{hr1+1}"

    # 02 Missing
    ws2=wb.create_sheet("02_Missing"); ws2.sheet_view.showGridLines=False
    _title(ws2,"MODULE 2 — MISSING VALUES",5); ws2.append([""])
    _note(ws2,"Period, Commodity, Province and Country must not be missing.",5); ws2.append([""])
    hr2=_hdr_row(ws2,["Column","Missing Count","% of Total","Status","Note"],[22,18,14,14,50])
    for i,(_, r) in enumerate(missing_df.iterrows()):
        _data_row(ws2,[r["Column"],r["Missing"],r["% of Total"],r["Status"],r["Note"]],alt=i%2==1,status_col=4)
    ws2.freeze_panes=f"A{hr2+1}"

    # 03 Duplicates
    ws3=wb.create_sheet("03_Duplicates"); ws3.sheet_view.showGridLines=False
    _title(ws3,"MODULE 3 — DUPLICATE DETECTION",6); ws3.append([""])
    hr3=_hdr_row(ws3,["Key Columns","Total Rows","Duplicate Rows","% of Total","Status","Note"],[40,14,18,14,14,44])
    for i,(_, r) in enumerate(dup_sum.iterrows()):
        _data_row(ws3,[r["Key Columns"],r["Total Rows"],r["Duplicate Rows"],r["% of Total"],r["Status"],r["Note"]],status_col=5)
    if not dup_ex.empty:
        ws3.append([""]); _note(ws3,"Sample duplicate rows (up to 50 shown):",6)
        ws3.append([""]); _hdr_row(ws3,list(dup_ex.columns),[20]*len(dup_ex.columns))
        for i,(_, r) in enumerate(dup_ex.iterrows()): _data_row(ws3,list(r),alt=i%2==1)
    ws3.freeze_panes=f"A{hr3+1}"

    # 04 Values
    ws4=wb.create_sheet("04_Values"); ws4.sheet_view.showGridLines=False
    _title(ws4,"MODULE 4 — VALUE & QUANTITY CONSISTENCY",8); ws4.append([""])
    _note(ws4,"Zero Qty rows are split into two categories: Blank unit (expected by Statistics Canada design) and Real unit (potentially suspicious).",8)
    ws4.append([""])
    hr4=_hdr_row(ws4,["Check","Count","% of Total","Status","Note"],[36,14,14,14,60])
    for i,(_, r) in enumerate(values_df.iterrows()):
        _data_row(ws4,[r["Check"],r["Count"],r["% of Total"],r["Status"],r["Note"]],alt=i%2==1,status_col=4)
    ws4.freeze_panes=f"A{hr4+1}"

    # Commodities with Zero Qty + Real Unit + Value > 0
    if not zero_top10.empty:
        ws4.append([""])
        _note(ws4,"COMMODITIES WITH ZERO QTY, REAL UNIT, AND VALUE > 0",8)
        _note(ws4,"These rows have export revenue recorded but Quantity = 0. Unit price cannot be calculated (÷ zero) so they are excluded from price analysis. Likely data entry errors — quantity was not entered.",8)
        ws4.append([""])
        top_cols=list(zero_top10.columns)
        top_widths=[55,22,22,18,12,30,34,28]
        _hdr_row(ws4,top_cols,top_widths[:len(top_cols)])
        flag_col=top_cols.index("Unit Flag")+1 if "Unit Flag" in top_cols else None
        for i,(_, r) in enumerate(zero_top10.iterrows()):
            vals=list(r); ws4.append(vals); rn=ws4.max_row
            bg=ALT if i%2==1 else WHT
            for c,v in enumerate(vals,1):
                cell=ws4.cell(rn,c)
                cell.border=_border(); cell.alignment=_left()
                if c==flag_col and isinstance(v,str) and "very suspicious" in v:
                    cell.fill=_fill(WN_B); cell.font=_font(bold=True,color=WN_F)
                else:
                    cell.fill=_fill(bg); cell.font=_font()

    # Individual example rows
    if not zero_susp.empty:
        ws4.append([""])
        _note(ws4,f"EXAMPLE ROWS ({len(zero_susp)} shown) — Sorted by commodity then value descending",8)
        ws4.append([""])
        _hdr_row(ws4,list(zero_susp.columns),[55,14,22,12,24,14,28])
        for i,(_, r) in enumerate(zero_susp.iterrows()):
            _data_row(ws4,list(r),alt=i%2==1)

    # 05 Units
    ws5=wb.create_sheet("05_Units"); ws5.sheet_view.showGridLines=False
    _title(ws5,"MODULE 5 — UNIT OF MEASURE CONSISTENCY",5); ws5.append([""])
    hr5=_hdr_row(ws5,["Total Commodities","Multi-Unit Commodities","% Affected","Status","Note"],[22,26,14,14,54])
    for i,(_, r) in enumerate(unit_sum.iterrows()):
        _data_row(ws5,[r["Total Commodities"],r["Multi-Unit Commodities"],r["% Affected"],r["Status"],r["Note"]],status_col=4)
    if not unit_det.empty:
        ws5.append([""]); _note(ws5,"Commodities with more than one unit (up to 100 shown):",4)
        ws5.append([""]); _hdr_row(ws5,["Commodity","Unit","Rows","Distinct Units"],[65,35,12,16])
        for i,(_, r) in enumerate(unit_det.iterrows()):
            _data_row(ws5,[r["Commodity"],r["Unit"],r["Rows"],r["Distinct Units"]],alt=i%2==1)
    ws5.freeze_panes=f"A{hr5+1}"

    REPORTS_DIR.mkdir(parents=True,exist_ok=True)
    wb.save(OUT_FILE)

if __name__=="__main__":
    print("\n"+"="*55)
    print("  REPORT 1 — DATA QUALITY  (Modules 1–5)")
    print("="*55)
    files=glob.glob(str(EXPORTS_DIR/"**/*.csv"),recursive=True)
    print(f"\n  Found {len(files)} CSV files in {EXPORTS_DIR}")
    if not files:
        print("  ERROR: No CSV files found. Check EXPORTS_DIR path."); exit(1)
    print("  Loading all files (this may take a minute)...")
    raw,errors=load_all(files)
    print(f"  Loaded {len(raw):,} rows from {raw['_source_file'].nunique() if not raw.empty else 0} files")
    if errors:
        print(f"  {len(errors)} files failed to load:")
        for e in errors: print(f"    {e['file']}: {e['error']}")
    print("  Running checks and building Excel report...")
    build(raw,errors)
    print(f"\n  ✅ Saved: {OUT_FILE}\n")


  REPORT 1 — DATA QUALITY  (Modules 1–5)

  Found 158 CSV files in /Users/midorikawaguti/DevProject/Dashboard-BR-CA/Dataset/Exports
  Loading all files (this may take a minute)...
  Loaded 2,562,365 rows from 158 files
  Running checks and building Excel report...

  ✅ Saved: /Users/midorikawaguti/DevProject/Dashboard-BR-CA/Reports/data_quality_report.xlsx

